# SQL Server RDS Test with mssql-python
Connect to AWS RDS SQL Server, create a test table, insert and update rows.

In [0]:
%pip install mssql-python --find-links /Volumes/shao_sandbox1/abac/mssql_python_wheels --no-index
dbutils.library.restartPython()

Looking in links: /Volumes/shao_sandbox1/abac/mssql_python_wheels
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Step 1: Check outbound IP (needed for RDS security group)
import urllib.request
my_ip = urllib.request.urlopen("https://ifconfig.me").read().decode()
print(f"Outbound IP: {my_ip}")
print(f"Make sure this IP is allowed in the RDS security group on port 1433")

Outbound IP: 163.228.235.47
Make sure this IP is allowed in the RDS security group on port 1433


In [0]:
# Step 2: Connect to SQL Server RDS
from mssql_python import connect

conn = connect(
    "Server=sqlserver-test.c7ysy68emu8t.us-west-2.rds.amazonaws.com,1433;"
    "UID=admin;"
    "PWD=TestPass1234;"
    "Database=master;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes",
    autocommit=True
)
cursor = conn.cursor()
print("Connected to SQL Server successfully!")

Connected to SQL Server successfully!


In [0]:
# Step 3: Create a test database and table
cursor.execute("IF DB_ID('testdb') IS NULL CREATE DATABASE testdb")
cursor.execute("USE testdb")

cursor.execute("""
IF OBJECT_ID('dbo.employees', 'U') IS NOT NULL
    DROP TABLE dbo.employees
""")

cursor.execute("""
CREATE TABLE dbo.employees (
    id INT IDENTITY(1, 1) PRIMARY KEY,
    name NVARCHAR(100),
    department NVARCHAR(50),
    salary DECIMAL(10, 2)
)
""")
print("Table 'employees' created in testdb")

Table 'employees' created in testdb


In [0]:
# Step 4: Insert rows
employees = [
    ("Alice", "Engineering", 120000),
    ("Bob", "Marketing", 95000),
    ("Charlie", "Engineering", 110000),
    ("Diana", "Sales", 88000),
    ("Eve", "Marketing", 92000),
]

for name, dept, salary in employees:
    cursor.execute(
        "INSERT INTO dbo.employees (name, department, salary) VALUES (?, ?, ?)",
        (name, dept, salary),
    )
print(f"Inserted {len(employees)} rows")

# Verify
cursor.execute("SELECT * FROM dbo.employees")
for row in cursor.fetchall():
    print(row)

Inserted 5 rows
(1, 'Alice', 'Engineering', 120000.00)
(2, 'Bob', 'Marketing', 95000.00)
(3, 'Charlie', 'Engineering', 110000.00)
(4, 'Diana', 'Sales', 88000.00)
(5, 'Eve', 'Marketing', 92000.00)


In [0]:
# Step 5: Update rows — give Engineering a 10% raise
cursor.execute("""
UPDATE dbo.employees
SET salary = salary * 1.10
WHERE department = 'Engineering'
""")
print(f"Updated {cursor.rowcount} rows (Engineering 10% raise)")

# Verify
cursor.execute("SELECT * FROM dbo.employees ORDER BY id")
for row in cursor.fetchall():
    print(row)

Updated 2 rows (Engineering 10% raise)
(1, 'Alice', 'Engineering', 132000.00)
(2, 'Bob', 'Marketing', 95000.00)
(3, 'Charlie', 'Engineering', 121000.00)
(4, 'Diana', 'Sales', 88000.00)
(5, 'Eve', 'Marketing', 92000.00)


In [0]:
# Cleanup
cursor.close()
conn.close()
print("Connection closed")

Connection closed
